In [6]:
# from google.colab import files
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score
import joblib

In [7]:
#upload = files.upload()

In [8]:
df = pd.read_csv("student-lifestyle-and-stress-dataset.csv")
print(df.head())
print(df['Student_Type'].value_counts())

  Student_Type  Sleep_Hours  Study_Hours  Social_Media_Hours  Attendance  \
0       school     6.868702     1.711722            3.176942         NaN   
1       school     8.519088     3.251084            3.880787   93.978465   
2      college     4.498770     6.306885            2.936172   64.421253   
3       school     8.591223     2.384922            5.222832   81.868960   
4      college     5.329293     9.345179            7.815869   85.847982   

   Exam_Pressure  Family_Support  Month  Stress_Level  
0            8.0             7.0    2.0             1  
1            6.0             4.0    3.0             1  
2            7.0             1.0   12.0             1  
3            2.0             7.0    7.0             0  
4            5.0             6.0   10.0             1  
Student_Type
college            13391
school              6021
working_student     4836
Name: count, dtype: int64


In [9]:
df.isnull().sum()
df = df.drop_duplicates()

In [10]:
X = df.copy()
X = X.drop(['Stress_Level', 'Month'], axis=1)

y = df['Stress_Level']

X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train)

          Student_Type  Sleep_Hours  Study_Hours  Social_Media_Hours  \
2110   working_student     7.573591     2.689889            6.807875   
18855              NaN     7.534359     3.308266            3.506916   
15115          college     5.732817     5.258596            3.256181   
19327           school     7.394586     3.054000            3.967751   
8358               NaN     6.760863     3.211396            1.063137   
...                ...          ...          ...                 ...   
21589          college     7.743528     4.616987            0.629008   
5393   working_student     7.020507     5.626361            6.791079   
860             school     7.555475     5.346715            6.191628   
15806           school     8.359162     3.134845            5.101058   
23670           school          NaN     6.482287            5.489750   

       Attendance  Exam_Pressure  Family_Support  
2110    80.742061            6.0             8.0  
18855   80.825796            4.0 

In [11]:
p1 = Pipeline(
    steps=[
        ('1', SimpleImputer(strategy='mean', ))
    ]
)
p2 = Pipeline(
    steps=[
        ('2', SimpleImputer(strategy='median'))
    ]
)

p_cat_student_type = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[['school', 'college', 'working_student']]))
])

In [12]:
ct = ColumnTransformer(
    transformers=[
        ('num_mean_imputer', p1, ['Sleep_Hours', 'Study_Hours', 'Social_Media_Hours', 'Attendance']),
        ('num_median_imputer', p2, ['Exam_Pressure', 'Family_Support']),
        ('cat_student_type_processor', p_cat_student_type, ['Student_Type'])
    ]
)

In [13]:
cleaner = Pipeline(
    steps=[
        ('cleaner', ct)
    ]
)

In [32]:
joblib.dump(cleaner, 'cleaner.pkl')

['D:\\c programs\\machine_learning_algorithms\\cleaner.pkl']

In [14]:
model = Pipeline(
    steps=[
        ('cleaning', ct),
        ('model', RandomForestClassifier(n_estimators=500, max_features=int((len(X_train.columns))**0.5)))
    ]
)

In [15]:
model.fit(X_train, Y_train)
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(Y_test, y_pred)*100}%")

Accuracy: 81.16169544740973%


In [17]:
joblib.dump(model, 'stress_model.pkl')

['C:\\Users\\dell\\Downloads\\stress_model.pkl']